# UMAP per-slide explorer for eval Task A2

This notebook reads the UMAP coordinate tables already saved by `src_A2/umap_utils.py`:

```text
<eval_root>/<checkpoint_name>/plots/<slide_id>_umap_coords.csv
```

It does **not** recompute UMAP. It only re-plots the stored coordinates with flexible highlighting:

- keep the classical four-panel layout: H&E/reference by modality, H&E/reference by cell type, GT/reference by modality, GT/reference by cell type;
- fade non-selected cell types/modalities into a transparent grey background;
- add one extra subplot per selected cell type, showing `reference + he + gt` together, colored by modality for that cell type;
- optionally use predicted labels from `predictions/<slide_id>.csv` for H&E/GT query cells.


In [ ]:
from pathlib import Path
from typing import Iterable, Optional, Sequence, Union, Mapping
import math
import warnings
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:  # no cover
    display = print

# Add the project root to sys.path so that 'src' can be imported
project_root = Path.cwd().parents[4]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.constants import get_full_path

In [ ]:
# ---------------------------------------------------------------------
# Loading helpers
# ---------------------------------------------------------------------

MODALITY_ALIASES = {
    "ref": "reference",
    "reference": "reference",
    "references": "reference",
    "scrna": "reference",
    "scrnaseq": "reference",
    "he": "he",
    "h&e": "he",
    "h_e": "he",
    "histology": "he",
    "gt": "gt",
    "ground_truth": "gt",
    "groundtruth": "gt",
    "st": "gt",
}

MODALITY_ORDER = ["reference", "he", "gt"]

# Fixed colors are useful here because modality color should stay stable across slides.
MODALITY_PALETTE = {
    "reference": "#4C78A8",
    "he": "#F58518",
    "gt": "#54A24B",
}


def _as_list(x):
    if x is None:
        return None
    if isinstance(x, str):
        return [x]
    return list(x)


def _normalize_modalities(modalities: Optional[Sequence[str]]) -> list[str]:
    if modalities is None:
        return MODALITY_ORDER.copy()
    out = []
    for m in _as_list(modalities):
        key = str(m).strip().lower()
        if key not in MODALITY_ALIASES:
            raise ValueError(
                f"Unknown modality {m!r}. Use any of: reference/ref, he/h&e, gt/st."
            )
        norm = MODALITY_ALIASES[key]
        if norm not in out:
            out.append(norm)
    return out


def list_umap_slides(checkpoint_dir: Union[str, Path]) -> list[str]:
    """Return slide_ids for which a saved UMAP coordinate CSV exists."""
    checkpoint_dir = Path(checkpoint_dir)
    plots_dir = checkpoint_dir / "plots"

    def _is_real_umap_coord_file(p: Path) -> bool:
        """
        Exclude macOS AppleDouble sidecar files such as:
            ._breast_s0_umap_coords.csv

        These files can appear after copying results from macOS and should never
        be interpreted as real slide UMAP coordinate files.
        """
        name = p.name
        return (
            p.is_file()
            and name.endswith("_umap_coords.csv")
            and not name.startswith("._")
            and not name.startswith(".")
        )

    files = sorted(p for p in plots_dir.glob("*_umap_coords.csv") if _is_real_umap_coord_file(p))
    return [p.name.removesuffix("_umap_coords.csv") for p in files]


def load_slide_umap_coords(
    slide_id: str,
    checkpoint_dir: Union[str, Path],
    enrich_with_predictions: bool = True,
) -> pd.DataFrame:
    """
    Load one saved UMAP coordinate table.

    Parameters
    ----------
    slide_id:
        Slide identifier, matching `<slide_id>_umap_coords.csv`.
    checkpoint_dir:
        A.2 checkpoint output folder.
    enrich_with_predictions:
        If True and `predictions/<slide_id>.csv` exists, merge per-cell prediction
        metadata onto H&E/GT query rows through `cell_id_int`.

    Returns
    -------
    DataFrame with at least:
        x, y, label, modality, kind, cell_id_int,
        plot_label_gt, plot_label_predicted
    """
    checkpoint_dir = Path(checkpoint_dir)
    slide_id = str(slide_id)
    if slide_id.startswith("._") or slide_id.startswith("."):
        raise ValueError(f"Invalid slide_id {slide_id!r}: this looks like a macOS AppleDouble/hidden sidecar file, not a real slide id.")

    coord_file = checkpoint_dir / "plots" / f"{slide_id}_umap_coords.csv"
    if not coord_file.exists():
        available = list_umap_slides(checkpoint_dir)
        hint = f" Available slides include: {available[:10]}" if available else " No UMAP coordinate files found."
        raise FileNotFoundError(f"Missing coordinate file: {coord_file}.{hint}")

    df = pd.read_csv(coord_file)
    required = {"x", "y", "label", "modality"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"{coord_file} is missing required columns: {sorted(missing)}")

    df = df.copy()
    df["slide_id"] = str(slide_id)
    df["modality"] = df["modality"].astype(str).str.lower().map(lambda x: MODALITY_ALIASES.get(x, x))
    bad_modalities = sorted(set(df["modality"].dropna()) - set(MODALITY_ORDER))
    if bad_modalities:
        raise ValueError(f"Unexpected modality values in {coord_file}: {bad_modalities}")

    df["label"] = df["label"].astype(str)
    df["plot_label_gt"] = df["label"]
    df["plot_label_predicted"] = df["label"]

    if "kind" not in df.columns:
        df["kind"] = df["modality"].map({"reference": "reference", "he": "he_query", "gt": "gt_query"})

    if "cell_id_int" not in df.columns:
        df["cell_id_int"] = np.nan

    # Merge prediction metadata only for query rows. Reference rows have no cell_id_int.
    pred_file = checkpoint_dir / "predictions" / f"{slide_id}.csv"
    if enrich_with_predictions and pred_file.exists():
        pred = pd.read_csv(pred_file)
        keep_cols = [
            c for c in [
                "cell_id_int",
                "cell_id",
                "gt_label",
                "he_pred_label",
                "gt_pred_label",
                "he_pred_fine",
                "gt_pred_fine",
                "he_valid",
                "gt_valid",
                "matched_valid",
                "covered",
                "selected_alpha",
                "cos_he_gt",
                "jaccard_he_gt_neighbors",
            ]
            if c in pred.columns
        ]
        if "cell_id_int" in keep_cols:
            pred = pred[keep_cols].drop_duplicates("cell_id_int")
            df = df.merge(pred, on="cell_id_int", how="left", suffixes=("", "_pred_table"))

            # Default label mode remains GT/reference, matching the saved UMAP CSV.
            if "gt_label" in df.columns:
                is_query = df["modality"].isin(["he", "gt"])
                df.loc[is_query & df["gt_label"].notna(), "plot_label_gt"] = (
                    df.loc[is_query & df["gt_label"].notna(), "gt_label"].astype(str)
                )

            # Predicted label mode: reference stays reference label; query labels come from the matching path.
            if "he_pred_label" in df.columns:
                mask = (df["modality"] == "he") & df["he_pred_label"].notna() & (df["he_pred_label"].astype(str) != "")
                df.loc[mask, "plot_label_predicted"] = df.loc[mask, "he_pred_label"].astype(str)
            if "gt_pred_label" in df.columns:
                mask = (df["modality"] == "gt") & df["gt_pred_label"].notna() & (df["gt_pred_label"].astype(str) != "")
                df.loc[mask, "plot_label_predicted"] = df.loc[mask, "gt_pred_label"].astype(str)

    return df


def summarize_slide_umap(
    slide_id: str,
    checkpoint_dir: Union[str, Path],
    label_mode: str = "gt_or_reference",
) -> pd.DataFrame:
    """Small count table to see which labels/modalities are present in the saved UMAP sample."""
    df = load_slide_umap_coords(slide_id, checkpoint_dir=checkpoint_dir)
    label_col = _resolve_label_column(df, label_mode)
    tab = (
        df.groupby(["modality", label_col], dropna=False)
        .size()
        .rename("n_points")
        .reset_index()
        .sort_values(["modality", "n_points"], ascending=[True, False])
    )
    return tab


In [ ]:
# ---------------------------------------------------------------------
# Plotting helpers
# ---------------------------------------------------------------------

def _resolve_label_column(df: pd.DataFrame, label_mode: str) -> str:
    """
    Choose which cell-type label to use for coloring/focusing.

    label_mode choices:
      - "gt_or_reference" / "stored": use the label saved in the UMAP CSV
        (reference mapped label, H&E/GT query ground-truth label).
      - "predicted": use H&E predicted label for H&E rows and GT-anchor predicted
        label for GT rows, if `predictions/<slide_id>.csv` was merged; reference
        rows still use the reference mapped label.
      - any existing dataframe column name, for custom analyses.
    """
    lm = str(label_mode).strip()
    aliases = {
        "stored": "plot_label_gt",
        "gt": "plot_label_gt",
        "ground_truth": "plot_label_gt",
        "gt_or_reference": "plot_label_gt",
        "pred": "plot_label_predicted",
        "predicted": "plot_label_predicted",
        "prediction": "plot_label_predicted",
    }
    col = aliases.get(lm, lm)
    if col not in df.columns:
        raise KeyError(
            f"label_mode={label_mode!r} resolved to column {col!r}, but this column is absent. "
            f"Available columns include: {sorted(df.columns)[:40]}"
        )
    return col


def _make_label_palette(labels: Sequence[str], cmap_name: str = "tab20") -> dict[str, object]:
    labels = [str(x) for x in labels]
    labels = list(dict.fromkeys(labels))
    if not labels:
        return {}

    # tab20 is readable for common cell-type counts. For many labels, sample a broader continuous cmap.
    cmap = plt.get_cmap(cmap_name if len(labels) <= 20 else "nipy_spectral")
    denom = max(1, len(labels) - 1)
    return {lab: cmap(i / denom) for i, lab in enumerate(labels)}


def _dot_size_for(modality: str, dot_size, default: float = 6.0) -> float:
    if isinstance(dot_size, Mapping):
        if modality in dot_size:
            return float(dot_size[modality])
        if "default" in dot_size:
            return float(dot_size["default"])
        return float(default)
    return float(dot_size)


def _limits_with_padding(df: pd.DataFrame, pad_fraction: float = 0.04):
    xmin, xmax = float(df["x"].min()), float(df["x"].max())
    ymin, ymax = float(df["y"].min()), float(df["y"].max())
    xpad = max(1e-6, (xmax - xmin) * pad_fraction)
    ypad = max(1e-6, (ymax - ymin) * pad_fraction)
    return (xmin - xpad, xmax + xpad), (ymin - ypad, ymax + ypad)


def _scatter_background(
    ax,
    df: pd.DataFrame,
    dot_size,
    background_color: str,
    background_alpha: float,
):
    if df.empty:
        return
    # Draw by modality only to keep optional modality-specific point sizes.
    for modality in MODALITY_ORDER:
        sub = df.loc[df["modality"] == modality]
        if sub.empty:
            continue
        ax.scatter(
            sub["x"],
            sub["y"],
            s=_dot_size_for(modality, dot_size, default=5.0),
            c=background_color,
            alpha=float(background_alpha),
            linewidths=0,
            rasterized=True,
        )


def _scatter_foreground(
    ax,
    df: pd.DataFrame,
    color_col: str,
    palette: Mapping[str, object],
    group_order: Sequence[str],
    dot_size,
    focus_alpha: float,
):
    handles = []
    labels = []
    if df.empty:
        return handles, labels

    for group in group_order:
        group = str(group)
        gdf = df.loc[df[color_col].astype(str) == group]
        if gdf.empty:
            continue
        color = palette.get(group, "black")
        # Draw by modality only to keep optional modality-specific point sizes.
        first_handle = None
        for modality in MODALITY_ORDER:
            sub = gdf.loc[gdf["modality"] == modality]
            if sub.empty:
                continue
            h = ax.scatter(
                sub["x"],
                sub["y"],
                s=_dot_size_for(modality, dot_size, default=8.0),
                c=[color],
                alpha=float(focus_alpha),
                linewidths=0,
                rasterized=True,
            )
            if first_handle is None:
                first_handle = h
        if first_handle is not None:
            handles.append(first_handle)
            labels.append(group)
    return handles, labels


def _finish_axis(ax, title: str, xlim, ylim):
    ax.set_title(title, fontsize=11)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal", adjustable="box")
    for spine in ax.spines.values():
        spine.set_visible(False)


def _add_legend(ax, handles, labels, max_legend_items: int = 18, legend: bool = True):
    if not legend or not handles:
        return
    if len(labels) > max_legend_items:
        ax.text(
            0.02,
            0.98,
            f"{len(labels)} colored groups\nlegend hidden",
            transform=ax.transAxes,
            va="top",
            ha="left",
            fontsize=8,
            bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "alpha": 0.75, "edgecolor": "none"},
        )
        return

    ncol = 1 if len(labels) <= 10 else 2
    ax.legend(
        handles,
        labels,
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        bbox_transform=ax.transAxes,
        fontsize=7,
        markerscale=1.8,
        frameon=False,
        handletextpad=0.2,
        borderaxespad=0.0,
        ncol=ncol,
        columnspacing=0.8,
    )


def _plot_panel(
    ax,
    df: pd.DataFrame,
    modalities_in_panel: Sequence[str],
    color_col: str,
    palette: Mapping[str, object],
    group_order: Sequence[str],
    focus_cell_types: Optional[Sequence[str]],
    focus_modalities: Sequence[str],
    title: str,
    xlim,
    ylim,
    dot_size,
    focus_alpha: float,
    background_alpha: float,
    background_color: str,
    legend: bool,
    max_legend_items: int,
):
    modalities_in_panel = list(modalities_in_panel)
    sub = df.loc[df["modality"].isin(modalities_in_panel)].copy()
    if sub.empty:
        _finish_axis(ax, title + "\n(no points)", xlim, ylim)
        return

    focus_mask = sub["modality"].isin(focus_modalities)
    if focus_cell_types is not None:
        wanted = set(map(str, focus_cell_types))
        focus_mask = focus_mask & sub[color_col].astype(str).isin(wanted)

    bg = sub.loc[~focus_mask]
    fg = sub.loc[focus_mask]

    _scatter_background(ax, bg, dot_size, background_color, background_alpha)
    handles, labels = _scatter_foreground(
        ax,
        fg,
        color_col=color_col,
        palette=palette,
        group_order=group_order,
        dot_size=dot_size,
        focus_alpha=focus_alpha,
    )
    suffix = "" if not fg.empty else "\n(no focused points)"
    _finish_axis(ax, title + suffix, xlim, ylim)
    _add_legend(ax, handles, labels, max_legend_items=max_legend_items, legend=legend)


In [ ]:
def plot_slide_umap(
    checkpoint_dir: Union[str, Path],
    slide_id: str,
    focus_cell_types: Optional[Union[str, Sequence[str]]] = None,
    focus_modalities: Optional[Sequence[str]] = ("reference", "he", "gt"),
    label_mode: str = "gt_or_reference",
    dot_size: Union[float, Mapping[str, float]] = 6.0,
    focus_alpha: float = 0.85,
    background_alpha: float = 0.05,
    background_color: str = "0.72",
    show_celltype_panels: bool = True,
    celltype_panel_modalities: Optional[Sequence[str]] = ("reference", "he", "gt"),
    legend: bool = True,
    max_legend_items: int = 18,
    label_cmap: str = "tab20",
    figsize_per_panel: tuple[float, float] = (5, 3),
    save_path: Optional[Union[str, Path]] = None,
    dpi: int = 180,
    enrich_with_predictions: bool = True,
    return_data: bool = False,
):
    """
    Plot one slide's saved UMAP with focus-aware transparency.

    Parameters
    ----------
    slide_id:
        Slide identifier to read from `<checkpoint_dir>/plots/<slide_id>_umap_coords.csv`.

    focus_cell_types:
        Cell type(s) to emphasize. Other cell types are drawn in transparent grey.
        Use None to emphasize all cell types in the classical panels.
        Use "all" to create one extra cell-type panel for every label in the UMAP sample.

    focus_modalities:
        Modalities to emphasize among {"reference", "he", "gt"}. Non-selected
        modalities are drawn in transparent grey.

    label_mode:
        "gt_or_reference" uses the saved labels: reference mapped labels and query
        GT labels. "predicted" uses H&E/GT predicted labels for query cells if
        the prediction table is available. You can also pass any column name in
        the loaded dataframe.

    dot_size:
        Either one scalar size or a dict, e.g.
        {"reference": 4, "he": 10, "gt": 10, "default": 6}.

    show_celltype_panels:
        If True, add one extra panel per selected cell type. In each such panel,
        that cell type is colored by modality: reference / he / gt. All other
        points are grey.

    return_data:
        If True, return `(fig, df)` instead of only `fig`.
    """
    checkpoint_dir = Path(checkpoint_dir)
    focus_modalities = _normalize_modalities(focus_modalities)
    celltype_panel_modalities = _normalize_modalities(celltype_panel_modalities)

    df = load_slide_umap_coords(
        slide_id=slide_id,
        checkpoint_dir=checkpoint_dir,
        enrich_with_predictions=enrich_with_predictions,
    )
    label_col = _resolve_label_column(df, label_mode)
    df = df.copy()
    df["_active_label"] = df[label_col].astype(str)
    df["_modality_label"] = pd.Categorical(df["modality"], categories=MODALITY_ORDER, ordered=True).astype(str)

    # Focus cell type normalization.
    if isinstance(focus_cell_types, str) and focus_cell_types.strip().lower() == "all":
        focus_cell_types_norm = sorted(df["_active_label"].dropna().astype(str).unique().tolist())
    else:
        focus_cell_types_norm = _as_list(focus_cell_types)
        if focus_cell_types_norm is not None:
            focus_cell_types_norm = [str(x) for x in focus_cell_types_norm]

    present_labels = set(df["_active_label"].astype(str).unique())
    if focus_cell_types_norm is not None:
        missing = [ct for ct in focus_cell_types_norm if ct not in present_labels]
        if missing:
            warnings.warn(
                f"These focus_cell_types are not present in the saved UMAP sample for slide {slide_id}: {missing}",
                stacklevel=2,
            )

    # Palette for cell-type panels and label-colored classical panels.
    if focus_cell_types_norm is None:
        label_order = sorted(df["_active_label"].dropna().astype(str).unique().tolist())
    else:
        label_order = [ct for ct in focus_cell_types_norm if ct in present_labels]
    # Include all labels in the palette so colors stay stable if you later remove focus_cell_types.
    all_labels = sorted(df["_active_label"].dropna().astype(str).unique().tolist())
    label_palette = _make_label_palette(all_labels, cmap_name=label_cmap)

    xlim, ylim = _limits_with_padding(df)

    extra_celltype_panels = []
    if show_celltype_panels and focus_cell_types_norm is not None:
        extra_celltype_panels = [ct for ct in focus_cell_types_norm if ct in present_labels]

    n_panels = 4 + len(extra_celltype_panels)
    ncols = 2
    nrows = int(math.ceil(n_panels / ncols))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(figsize_per_panel[0] * ncols, figsize_per_panel[1] * nrows),
        squeeze=False,
        constrained_layout=True,
    )
    fig.set_constrained_layout_pads(w_pad=0.08, h_pad=0.06, wspace=0.08, hspace=0.08)
    axes_flat = axes.ravel()

    # Classical panel A: H&E + reference, colored by modality.
    _plot_panel(
        axes_flat[0],
        df,
        modalities_in_panel=["reference", "he"],
        color_col="_modality_label",
        palette=MODALITY_PALETTE,
        group_order=["reference", "he"],
        focus_cell_types=None,
        focus_modalities=focus_modalities,
        title=f"{slide_id} A · H&E + reference by modality",
        xlim=xlim,
        ylim=ylim,
        dot_size=dot_size,
        focus_alpha=focus_alpha,
        background_alpha=background_alpha,
        background_color=background_color,
        legend=legend,
        max_legend_items=max_legend_items,
    )

    # Classical panel B: H&E + reference, colored by mapped/active label.
    _plot_panel(
        axes_flat[1],
        df,
        modalities_in_panel=["reference", "he"],
        color_col="_active_label",
        palette=label_palette,
        group_order=label_order if focus_cell_types_norm is not None else all_labels,
        focus_cell_types=focus_cell_types_norm,
        focus_modalities=focus_modalities,
        title=f"{slide_id} B · H&E + reference by cell type",
        xlim=xlim,
        ylim=ylim,
        dot_size=dot_size,
        focus_alpha=focus_alpha,
        background_alpha=background_alpha,
        background_color=background_color,
        legend=legend,
        max_legend_items=max_legend_items,
    )

    # Classical panel C: GT + reference, colored by modality.
    _plot_panel(
        axes_flat[2],
        df,
        modalities_in_panel=["reference", "gt"],
        color_col="_modality_label",
        palette=MODALITY_PALETTE,
        group_order=["reference", "gt"],
        focus_cell_types=None,
        focus_modalities=focus_modalities,
        title=f"{slide_id} C · GT + reference by modality",
        xlim=xlim,
        ylim=ylim,
        dot_size=dot_size,
        focus_alpha=focus_alpha,
        background_alpha=background_alpha,
        background_color=background_color,
        legend=legend,
        max_legend_items=max_legend_items,
    )

    # Classical panel D: GT + reference, colored by mapped/active label.
    _plot_panel(
        axes_flat[3],
        df,
        modalities_in_panel=["reference", "gt"],
        color_col="_active_label",
        palette=label_palette,
        group_order=label_order if focus_cell_types_norm is not None else all_labels,
        focus_cell_types=focus_cell_types_norm,
        focus_modalities=focus_modalities,
        title=f"{slide_id} D · GT + reference by cell type",
        xlim=xlim,
        ylim=ylim,
        dot_size=dot_size,
        focus_alpha=focus_alpha,
        background_alpha=background_alpha,
        background_color=background_color,
        legend=legend,
        max_legend_items=max_legend_items,
    )

    # Extra targeted panels: one per requested cell type, colored by modality.
    for i, cell_type in enumerate(extra_celltype_panels, start=4):
        sub = df.loc[df["modality"].isin(celltype_panel_modalities)].copy()
        focus_mask = sub["_active_label"].astype(str).eq(str(cell_type)) & sub["modality"].isin(focus_modalities)
        bg = sub.loc[~focus_mask]
        fg = sub.loc[focus_mask]

        ax = axes_flat[i]
        _scatter_background(ax, bg, dot_size, background_color, background_alpha)
        handles, labels = _scatter_foreground(
            ax,
            fg,
            color_col="_modality_label",
            palette=MODALITY_PALETTE,
            group_order=[m for m in MODALITY_ORDER if m in celltype_panel_modalities],
            dot_size=dot_size,
            focus_alpha=focus_alpha,
        )
        suffix = "" if not fg.empty else "\n(no focused points)"
        _finish_axis(
            ax,
            f"{slide_id} · {cell_type}\nreference + H&E + GT by modality" + suffix,
            xlim,
            ylim,
        )
        _add_legend(ax, handles, labels, max_legend_items=max_legend_items, legend=legend)

    # Hide unused axes.
    for j in range(n_panels, len(axes_flat)):
        axes_flat[j].axis("off")

    focus_ct_text = "all cell types" if focus_cell_types_norm is None else ", ".join(focus_cell_types_norm)
    fig.suptitle(
        f"slide={slide_id} · labels={label_mode} · \nfocus_modalities={focus_modalities} · focus_cell_types={focus_ct_text}",
        fontsize=13,
    )

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=dpi, bbox_inches="tight")
        print(f"Saved: {save_path}")

    if return_data:
        return fig, df
    return fig


## eval_1 / t46e13clip


In [ ]:
eval_id = "eval_1"
training_num = 16
checkpoint_num = 31

In [ ]:
CHECKPOINT_DIR = get_full_path("eval", "A-2_eval_st_predictions", eval_id, f"HEencoder_t{training_num}e{checkpoint_num}_clip")
UMAP_DIR = CHECKPOINT_DIR / "plots"
PREDICTIONS_DIR = CHECKPOINT_DIR / "predictions"

In [ ]:
# Check which slide IDs have saved UMAP coordinates
slides = list_umap_slides(CHECKPOINT_DIR)
print(f"Found {len(slides)} slide(s).")
slides[:10]


In [ ]:
slide_id = "kidney_s0"

In [ ]:
# Inspect point counts by modality and cell type for one slide
counts = summarize_slide_umap(slide_id, CHECKPOINT_DIR)
#display(counts.head(40))

# List of cell types on one line (to copy into focus_cell_types)
cell_types = counts["plot_label_gt"].unique().tolist()
print("'" + "', '".join(str(ct) for ct in cell_types) + "'")

In [ ]:
# Focus
fig = plot_slide_umap(
    checkpoint_dir=CHECKPOINT_DIR,
    slide_id=slide_id,
    focus_cell_types=['Epithelial', 'Blood_vessel', 'Myeloid', 'Fibroblast_Myofibroblast', 'Unknown', 'T_NK', 'B_Plasma', 'Specialized'],             # use None to focus on all cell type or list of the cell type names to focus on specific ones
    focus_modalities=("reference", "he", "gt"),  # use all modalities or a subset, e.g. ("he", "gt") to focus only on query modalities, choices are "reference", "he", "gt"
    label_mode="gt_or_reference",                # "gt_or_reference" uses the reference mapped labels and query GT labels. "predicted" uses H&E/GT predicted labels for query cells
    dot_size={"reference": 2, "he": 2, "gt":2},
    focus_alpha=0.90,
    background_alpha=0.035,
    show_celltype_panels=True,                   # whether to add extra panels that each focus on one cell type, colored by modality
)
plt.show()
